<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_01_cnn_image_classification.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 01 — Finding Weld Defects with a CNN

**Deep Learning for Engineering · Aalborg University · Part 1**

## The problem

A radiographer looks at an X-ray film of a welded seam and sorts it into one of
three classes:

* **clean** — sound material;
* **crack** — a thin dark line;
* **pit** — a small, round dark spot (porosity).

In this notebook a network learns to do the same on 600 small radiographs, 16 by
16 pixels each. The images are drawn by the course code rather than measured — a
brightness ramp, film grain, and a dark line or spot — and no radiographer would
be fooled by them. They keep the one property that matters here: **a defect can
be anywhere on the plate, and it is the same defect wherever it is.**

Two networks try the task.

* The **baseline** is a dense feedforward network, as in L4: the image is
  flattened into 256 numbers and every pixel is connected to every neuron of a
  hidden layer.
* The **CNN** (convolutional neural network, L5.1) slides small 3 by 3 kernels
  over the image, with the same nine weights at every position.

The notebook goes in a straight line:

| section | what you do |
|---|---|
| 1 | define the CNN and count its parameters |
| 2 | set a convolution's weights by hand and see what each kernel picks out |
| 3 | train the CNN, and compare the weights it learned with the hand-set ones |
| 4 | train the dense baseline with the same recipe, and compare the two |
| 5 | save the results for the report |

There are three small `TODO` cells, one line each.

---

## 0 · Setup


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_5_core as core
core.keep_outputs()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

core.set_seed(0)

X, y = core.weld_images(n_per_class=200, seed=3)
X_train, y_train, X_test, y_test = core.split_data(X, y, frac=0.7, seed=5)

print("all      :", X.shape, " class counts", np.bincount(y))
print("training :", X_train.shape)
print("held out :", X_test.shape)

core.plot_images(X_train, y_train, n=12, title="Twelve of the training images")
plt.show()

**What you should see.** `all : (600, 1, 16, 16)` with class counts
`[200 200 200]`, 420 training images, 180 held out, and twelve labelled
radiographs. The 180 held-out images are never used for training; every accuracy
called *held-out* below is measured on them.

The `1` in `(600, 1, 16, 16)` is the number of **channels**: a greyscale image
has one. PyTorch's convolution layers expect `(batch, channels, height, width)`.

---

## 1 · The CNN and its parameters

The CNN alternates convolution, ReLU and pooling twice, and then one dense layer
makes the decision:

```
input                            (1, 16, 16)   one grey channel
Conv2d(1 -> 8, 3x3, padding=1)   (8, 16, 16)   eight feature maps
ReLU, MaxPool2d(2)               (8, 8, 8)
Conv2d(8 -> 16, 3x3, padding=1)  (16, 8, 8)    sixteen feature maps
ReLU, MaxPool2d(2)               (16, 4, 4)
Flatten                          (256,)
Linear(256 -> 3)                 (3,)          one score per class
```

* **`padding=1`** puts a border of zeros around the input, so that the 3 by 3
  kernel also fits over the edge pixels and a 16 by 16 input gives a 16 by 16
  feature map.
* **`MaxPool2d(2)`** keeps the largest value in every 2 by 2 block. It halves
  the size, and it forgets exactly where in the block the defect was — which is
  what a classifier wants.
* **Three outputs and no softmax.** `nn.CrossEntropyLoss` applies the softmax
  itself.

A convolution layer has $K^2 C_{\mathrm{in}} C_{\mathrm{out}} + C_{\mathrm{out}}$
parameters: kernel size $K$, input channels, output channels, and one bias per
output channel. **There is no image size in that count**, because the same
kernel is used at every position.

### Your turn — TODO 1

The kernel size and the two channel counts are defined at the top of the cell.
Fill in the second convolution and the final classifier.


In [ ]:
# The design choices, in one place.
K  = 3     # kernel size: every neuron looks at a 3 x 3 patch of its input
C1 = 8     # feature maps made by the first convolution
C2 = 16    # feature maps made by the second convolution
core.set_seed(0)

# TODO 1 --- the convolutional network ----------------------------------------------------
# Two `...` to replace:
#   second_conv  ->  nn.Conv2d(C1, C2, kernel_size=K, padding=1)   8 maps in, 16 maps out
#   classifier   ->  nn.Linear(C2 * 4 * 4, 3)                       16 maps of 4x4, three classes
second_conv = ...
classifier  = ...
assert not any(v is ... for v in (second_conv, classifier)), "TODO 1: replace the two ..."

cnn = nn.Sequential(
    nn.Conv2d(1, C1, kernel_size=K, padding=1),   # (1, 16, 16) -> (8, 16, 16)  eight 3x3 kernels
    nn.ReLU(),                                    # keep positive responses, zero the rest
    nn.MaxPool2d(2),                              # (8, 16, 16) -> (8, 8, 8)    largest of each 2x2 block
    second_conv,                                  # (8, 8, 8)   -> (16, 8, 8)   sixteen 3x3x8 kernels
    nn.ReLU(),
    nn.MaxPool2d(2),                              # (16, 8, 8)  -> (16, 4, 4)
    nn.Flatten(),                                 # (16, 4, 4)  -> (256,)
    classifier,                                   # (256,)      -> (3,)         one score per class
)
n_cnn = core.count_parameters(cnn)
# ------------------------------------------------------------------------------

In [ ]:
print("parameters:", n_cnn)
for name, p in cnn.named_parameters():
    print(f"  {name:10s} {str(tuple(p.shape)):>15s}  {p.numel():5d}")

with torch.no_grad():
    probe = cnn(torch.tensor(X_train[:4]))
print("\nshape check:", X_train[:4].shape, "->", tuple(probe.shape))

# One layer, two ways: eight 16x16 feature maps from one 16x16 image.
conv_16  = 3 * 3 * 1 * 8 + 8                  # eight 3x3 kernels and eight biases
dense_16 = (16 * 16) * (16 * 16 * 8) + 16 * 16 * 8   # every pixel to every output, plus biases
print(f"\none layer, 16x16 image -> eight 16x16 maps:"
      f"  convolution {conv_16:,}   dense {dense_16:,}")

**What you should see.** `parameters: 2019`, and the breakdown:

| layer | weights | biases | total |
|---|---|---|---|
| first convolution | $3 \times 3 \times 1 \times 8 = 72$ | 8 | **80** |
| second convolution | $3 \times 3 \times 8 \times 16 = 1152$ | 16 | **1168** |
| classifier | $256 \times 3 = 768$ | 3 | **771** |

then `shape check: (4, 1, 16, 16) -> (4, 3)`, and the one-layer comparison:
`convolution 80   dense 526,336`.

That last line is the argument for convolution in one number. Making eight
16 by 16 feature maps from a 16 by 16 image costs a convolution 80 parameters;
a dense layer producing the same 2048 outputs connects every one of them to all
256 pixels and needs over half a million. The convolution can do with 80 because
it assumes that the same small pattern means the same thing wherever it is on
the plate.

Run the shape check before training, every time: it catches a wrong flatten size
in one line.

---

## 2 · Hand-set weights

A convolution slides a 3 by 3 kernel over the image. At each position it
multiplies the nine pixels under the kernel by the nine weights and adds them up:

$$h_{ij} = \sum_{m}\sum_{n} k_{mn}\, x_{i+m,\,j+n}$$

Before any training, those nine weights can be chosen by hand. `core.KERNELS`
holds four classic choices:

* **vertical edge** — negative on the left, positive on the right: large where
  the brightness changes from left to right;
* **horizontal edge** — the same, from top to bottom;
* **blob** — $+8$ in the centre and $-1$ around it: large where a pixel differs
  from its neighbours;
* **blur** — all nine weights $1/9$: the local average.

### Your turn — TODO 2

Put the four kernels into a PyTorch `nn.Conv2d` layer by hand — the same kind of
layer the CNN uses, only here the weights are yours and not learned. The layer
has no padding, so a 16 by 16 image gives a 14 by 14 map and the edge of the
plate does not light up.

Predict first: which kernels should respond to the crack, and which to the pit?


In [ ]:
# TODO 2 --- hand-set weights ---------------------------------------------------------------
# One `...` to replace, inside the loop:
#   w  ->  torch.tensor(kernel, dtype=torch.float32)     the kernel's nine numbers as a tensor
hand = nn.Conv2d(1, 4, kernel_size=3, bias=False)   # four kernels, no padding, no bias
kernel_names = list(core.KERNELS)
with torch.no_grad():
    for i, kernel in enumerate(core.KERNELS.values()):
        w = ...
        assert w is not ..., "TODO 2: replace the ..."
        hand.weight[i, 0] = w                       # output map i, input channel 0
# ------------------------------------------------------------------------------

In [ ]:
for name, kernel in core.KERNELS.items():
    print(f"{name}:\n{np.round(kernel, 2)}\n")

# The first training image of each class, and its four feature maps.
examples = np.stack([X_train[y_train == c][0, 0] for c in range(3)])   # (3, 16, 16)
with torch.no_grad():
    maps = hand(torch.tensor(examples[:, None])).numpy()               # (3, 4, 14, 14)
print("images", examples.shape, "-> feature maps", maps.shape)

# The same four kernels on all 420 training images: the largest response
# (ignoring its sign) in each map, averaged over the images of each class.
with torch.no_grad():
    maps_all = hand(torch.tensor(X_train)).numpy()                     # (420, 4, 14, 14)
peak = np.abs(maps_all).max(axis=(2, 3))                                # (420, 4)
print("\nlargest response in a map, averaged over the images of each class")
print("           " + "".join(f"{n:>17s}" for n in kernel_names))
for c, cname in enumerate(core.CLASS_NAMES):
    print(f"  {cname:8s} " + "".join(f"{v:17.2f}" for v in peak[y_train == c].mean(axis=0)))

core.plot_responses(examples, maps, core.CLASS_NAMES, kernel_names,
                    title="Four hand-set kernels against three classes")
plt.show()

**What you should see.** The four kernels, `images (3, 16, 16) -> feature maps
(3, 4, 14, 14)`, this table,

```
               vertical edge  horizontal edge             blob             blur
  clean                 0.53             0.53             1.23             0.67
  crack                 1.29             1.29             2.27             0.67
  pit                   1.08             1.08             1.36             0.67
```

and a grid: one image of each class down the left, and its four feature maps to
the right.

* The two **edge** kernels do what they were chosen for. Their response on a
  cracked or pitted plate is about twice their response on a clean one. In the
  grid the crack is traced by a red line beside a blue one: the two sides of a
  dark line are a bright-to-dark and a dark-to-bright change.
* The **blob** kernel responds strongly to the crack, but hardly more to the pit
  than to a clean plate. Its $+8$ centre also amplifies the film grain, and a
  pit two or three pixels wide is wider than the kernel. A kernel chosen by hand
  for pits does not find these pits.
* The **blur** kernel gives the same response to every class. It smooths the
  image and separates nothing.

Choosing kernels by hand works when the thing to find is simple, and fails
quietly when it is not. The CNN keeps the idea — small kernels, applied
everywhere — and lets training choose the numbers.

### Why are the feature maps coloured?

The input is grey and the feature maps are red and blue, but **a feature map is
not a colour image**. It is one channel: a 14 by 14 array of plain numbers, the
same kind of array as the grey image. The colour is only a *colour map* that
matplotlib applies when it draws the array — a palette that turns each number
into a colour.

The image is drawn with a grey palette because its numbers are brightness,
from 0 (dark) to 1 (bright). A feature map's numbers are signed: positive where
the patch under the kernel looks like the kernel, negative where it looks like
its opposite, near zero where the kernel sees nothing. So it is drawn with a
diverging palette: **red for positive, blue for negative, white for zero**. The
colour bar under each column gives the scale.

Two consequences. The blur map is all red because every one of its numbers is
positive. And the word *channel* in a CNN does not mean a colour: the first
layer of this CNN makes eight channels from one grey image, and each of them is
another array of numbers like these.


In [ ]:
core.set_seed(0)
hist_cnn = core.train_classifier(cnn, X_train, y_train, X_test, y_test,
                                 epochs=1000, lr=1e-3)

print(f"training loss   : {hist_cnn['loss'][0]:.3f} at the start -> {hist_cnn['loss'][-1]:.4f} at the end")
print(f"train accuracy  : {hist_cnn['acc_train'][-1]:.3f}")
print(f"held-out acc.   : {hist_cnn['acc_test'][-1]:.3f}")
print(f"training time   : {hist_cnn['seconds']:.1f} s")

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(np.arange(1, 1001), hist_cnn["loss"], lw=1.6, color="#1f77b4", label="CNN, training loss")
ax.axhline(np.log(3), ls="--", lw=1.0, color="#777777", label="ln 3: guessing among three classes")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("cross entropy [nats]")
ax.set_title("Training the CNN"); ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** `training loss : 1.101 at the start -> 0.0026 at the
end`, `train accuracy : 1.000`, `held-out acc. : 0.989`, a training time in
seconds, and a loss curve that starts on the dashed line and falls steeply after
the first fifty epochs.

The loss starts at $\ln 3 = 1.099$ because an untrained network gives the three
classes about equal probability, and $-\ln \tfrac{1}{3} = \ln 3$. A three-class
cross entropy that starts anywhere else means that the labels, the shapes or the
initialisation are wrong.

---

## 3 · Learned weights

The first convolution of the trained CNN holds eight 3 by 3 kernels, found by
gradient descent instead of chosen by hand. The next cell draws them under the
four hand-set ones, and then shows the eight feature maps they make from one
cracked plate the network never trained on.


In [ ]:
W_learned = cnn[0].weight.detach().numpy()        # (8, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(13.0, 3.9))
for i in range(8):
    axes[0, i].axis("off")
for i, (name, kernel) in enumerate(core.KERNELS.items()):
    core.show_kernel(kernel, ax=axes[0, i], title=f"set: {name}")
for i in range(8):
    core.show_kernel(W_learned[i, 0], ax=axes[1, i], title=f"learned k{i}")
fig.suptitle("Hand-set kernels (top) and the CNN's eight learned first-layer kernels (bottom)",
             fontsize=12)
fig.tight_layout()
plt.show()

crack = X_test[y_test == 1][0]                   # one cracked plate the CNN never trained on
with torch.no_grad():
    maps_learned = cnn[0](torch.tensor(crack[None])).numpy()[0]   # (8, 16, 16)
core.plot_feature_maps(maps_learned, image=crack[0],
                       title="Learned first-layer feature maps for one cracked plate")
plt.show()

**What you should see.** Two rows of small kernels, the four hand-set ones on top
and the eight learned ones below, each cell with its weight written on it; then
the cracked plate and its eight feature maps.

The learned kernels do not look like the hand-set ones, and they need not.
Several are edge detectors at some angle — negative weights on one side,
positive on the other — because a crack at any angle and the rim of a pit are
both edges. One or two are positive almost everywhere, closer to the blur. None
is a tidy textbook kernel: nothing in the loss rewards a tidy kernel, only a
correct class.

The feature maps are easier to read than the kernels. Look for maps in which the
crack stands out as a red or blue streak against a pale background: those
channels have learned to respond to the crack. Two things are artefacts rather
than learning. A map that is uniformly red or blue is shifted by its bias. And
a thin frame around the edge comes from the zero padding, where the kernel sees a
"dark border" that is not on the plate. Each map here has its own colour scale,
written under it, because the eight maps differ in size.

---

## 4 · The dense baseline, and the comparison

The baseline flattens the image into 256 numbers and puts one hidden layer of 64
neurons on it: $256 \times 64 + 64 + 64 \times 3 + 3 = 16{,}643$ parameters.

It is trained with **exactly the same recipe** as the CNN: the same 420 images,
the same loss, Adam at the same learning rate, the same 1000 epochs, the same
function. 1000 epochs is enough for both networks to fit their training images,
so neither is short of training.

### Your turn — TODO 3

Fill in the hidden layer.


In [ ]:
core.set_seed(0)

# TODO 3 --- the dense baseline ------------------------------------------------------------
# One `...` to replace:
#   hidden  ->  nn.Linear(16 * 16, 64)      every one of the 256 pixels to each of 64 neurons
hidden = ...
assert hidden is not ..., "TODO 3: replace the ..."

dense = nn.Sequential(
    nn.Flatten(),          # (1, 16, 16) -> (256,)   the image as one long vector
    hidden,                # (256,) -> (64,)
    nn.ReLU(),
    nn.Linear(64, 3),      # (64,) -> (3,)           one score per class
)
n_mlp = core.count_parameters(dense)

# The same recipe as the CNN: same data, Adam, learning rate 0.001, 1000 epochs.
hist_mlp = core.train_classifier(dense, X_train, y_train, X_test, y_test,
                                 epochs=1000, lr=1e-3)
# ------------------------------------------------------------------------------

In [ ]:
acc_train, acc_test = float(hist_cnn["acc_train"][-1]), float(hist_cnn["acc_test"][-1])
acc_train_mlp, acc_test_mlp = float(hist_mlp["acc_train"][-1]), float(hist_mlp["acc_test"][-1])
time_cnn, time_mlp = hist_cnn["seconds"], hist_mlp["seconds"]

rows = [["CNN", f"{n_cnn:,}", f"{time_cnn:.1f}", f"{acc_train:.3f}", f"{acc_test:.3f}"],
        ["dense", f"{n_mlp:,}", f"{time_mlp:.1f}", f"{acc_train_mlp:.3f}", f"{acc_test_mlp:.3f}"]]
print(core.error_table(rows, ["model", "parameters", "training time [s]",
                              "train accuracy", "held-out accuracy"]))
print(f"\nthe dense network has {n_mlp / n_cnn:.1f} times as many parameters")

conf_cnn = core.confusion(y_test, hist_cnn["pred_test"])
conf_mlp = core.confusion(y_test, hist_mlp["pred_test"])
core.plot_confusions({"CNN": conf_cnn, "dense": conf_mlp},
                     title="Held-out images: true class against predicted class")
plt.show()

fig, ax = plt.subplots(figsize=(6.8, 4.0))
for hist, name, col in [(hist_cnn, "CNN", "#1f77b4"), (hist_mlp, "dense", "#d94f2b")]:
    ax.plot(hist["epoch"], hist["acc_train"], ls="--", lw=1.4, color=col, label=f"{name}, training images")
    ax.plot(hist["epoch"], hist["acc_test"], ls="-", lw=1.8, color=col, label=f"{name}, held-out images")
ax.set_ylim(0.3, 1.02); ax.set_xlabel("epoch"); ax.set_ylabel("accuracy [fraction correct]")
ax.set_title("Same data, same recipe"); ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.25)
plt.show()

**What you should see.**

| model | parameters | training time [s] | train accuracy | held-out accuracy |
|---|---|---|---|---|
| CNN | 2,019 | 3.4 | 1.000 | 0.989 |
| dense | 16,643 | 0.8 | 0.995 | 0.711 |

`the dense network has 8.2 times as many parameters`, two confusion matrices —
the CNN gets 178 of the 180 held-out images right, the dense network 128 — and
the accuracy curves of both networks.

The accuracies and counts are the same on every machine. The training times are
not: they depend on the computer and on what else it is running. The times above
were measured on one computer's CPU; on the same computer, busy with other work,
the CNN took 50 s and the dense network 1 s. What holds everywhere is that the CNN is the
slower of the two.

**The result.** The CNN classifies the held-out radiographs at least as well as
the dense baseline — in fact much better, 0.989 against 0.711 — with **eight
times fewer parameters**.

**Why.** Both networks fit their training images almost perfectly, so the dense
network is not short of capacity or of training. The difference is on the images
neither has seen. The dense network learns each pixel position separately, so it
has to learn "a crack at the top left" and "a crack at the bottom right" as two
different things, from the few training images that happen to have a crack
there. It memorises the training images instead: its training accuracy climbs to 1
while its held-out accuracy stops at about 0.7, and its confusion matrix shows
it mixing up cracks and pits. The CNN uses one kernel at every position, so a crack it has learned in one
place it recognises in every other.

**What the CNN does not save.** Training time. Each of its 2,019 weights is used
at every position of the image, so it does more arithmetic than the dense
network and trains several times slower. A convolution saves *parameters* —
memory, and the training data needed to fit them — not computation.

**When the result would change.** The defects here are equally likely anywhere
on the plate, which is exactly the assumption a convolution makes. If position
mattered — a defect that is only serious near the weld root, say — a convolution
would throw away information the dense network could use.

---

## 5 · Save

Notebook 05 reads this file.


In [ ]:
import os
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb01_cnn.npz")
np.savez(path,
         n_cnn=n_cnn, n_mlp=n_mlp,
         acc_train=acc_train, acc_test=acc_test,
         acc_train_mlp=acc_train_mlp, acc_test_mlp=acc_test_mlp,
         time_cnn=time_cnn, time_mlp=time_mlp,
         dense_16=dense_16, conv_16=conv_16,
         losses=hist_cnn["loss"], losses_mlp=hist_mlp["loss"],
         confusion=conf_cnn, confusion_mlp=conf_mlp)
print("wrote", path)
core.saved(path)


**What you should see.** `wrote .../Ex05_outputs/nb01_cnn.npz`.

---

## 6 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The first convolution of your CNN has 80 parameters; a dense layer making the same eight 16 by 16 feature maps would need 526,336. Where does each number come from, and why does the image size appear in one count but not in the other? What property of the weld images makes the smaller count safe to use?
   *→ L5.1 Q3, Q4*
2. Follow one radiograph through your CNN and write down its shape after every layer. What does the padding do to the size, what do the two pooling layers do, and what does pooling throw away that a classifier of clean, crack and pit does not need?
   *→ L5.1 Q2, Q3*
3. Your hand-set feature maps were drawn in red and blue, although the radiograph is grey. What is a feature map, and what do red, blue and white mean on it? Why did the blob kernel, chosen by hand to find pits, hardly separate pits from clean plate, and what does training do instead?
   *→ L5.1 Q1*
4. Both networks fitted their training images almost perfectly, yet on the held-out images the CNN scored 0.989 with 2,019 parameters and the dense network 0.711 with 16,643. Explain the difference to a colleague who says more parameters give a better model. What did the CNN not save, and when would the dense network be the better choice?
   *→ L5.1 Q4*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.
